## NEOs close-approaches — construcción del dataset y etiquetado PHA

Este proyecto estudia los Objetos Cercanos a la Tierra (NEOs) a partir de su
registro histórico de **aproximaciones cercanas** (1900–presente). Combina un
análisis exploratorio no supervisado (PCA, K-Means, t-SNE) con la pregunta
central del trabajo: **¿puede inferirse el carácter potencialmente peligroso
(PHA) de un NEO solo desde la cinemática de sus aproximaciones observadas, y
cómo distorsiona esa inferencia la función de selección observacional del
catálogo?**

Este notebook construye el dataset (descarga desde las APIs de JPL, limpieza y
etiquetado). El análisis vive en `notebooks/ProyectoNeoRework_ml.ipynb`.

# Data
Los datos fueron extraídos mediante el consumo de la API proporcionada por el sistema de monitoreo de aproximaciones cercanas a la Tierra (Close-Approach Data) del Center for Near Earth Object Studies, perteneciente a la NASA. La fuente oficial se encuentra disponible en: https://cneos.jpl.nasa.gov/ca/

## Librerias data

In [1]:
import os
import sys
import time
from datetime import datetime

import pandas as pd
import requests

# Utilidades compartidas con el notebook de ML y con scripts/ (paquete neos/ en
# la raiz del repo): constantes, descarga, limpieza y agregacion por objeto.
sys.path.insert(0, os.path.abspath(".."))
from neos import datos
from neos.constantes import DIR_DATOS, DIST_MAX_AU, RUTA_CAD, RUTA_SBDB

## Carga y preparacion de datos

In [2]:
# Las rutas salen de neos.constantes (absolutas, ancladas a la raiz del repo),
# asi que ya no dependen del directorio de trabajo del notebook.
os.makedirs(DIR_DATOS, exist_ok=True)
ruta_csv = RUTA_CAD

In [4]:
# Carga de datos
necesita_descarga = True
df = None

if os.path.exists(ruta_csv):
    print(f"✔ Archivo CSV encontrado en {ruta_csv}")
    if datos.archivo_es_reciente(ruta_csv, dias_maximos=30):
        print("✔ Archivo actualizado (menos de 30 días). Cargando desde archivo local...")
        try:
            df = pd.read_csv(ruta_csv)
            print(f"✔ Datos cargados correctamente. {len(df):,} registros.")
            df['Close-Approach (CA) Date'] = df['Close-Approach (CA) Date'].apply(datos.limpiar_fecha)
            necesita_descarga = False
        except Exception as e:
            print(f"✘ Error al cargar el archivo: {e}")
            print("  Se procederá a descargar datos frescos...")
            necesita_descarga = True
    else:
        print("✘ Archivo desactualizado (más de 30 días). Se descargará de nuevo.")
else:
    print(f"✘ Archivo CSV no encontrado en {ruta_csv}")

 # DESCARGA DESDE LA API DE JPL

if necesita_descarga:
    print("\n⬇ Descargando datos desde la API de JPL (~6 min, 339k eventos)...")

    try:
        # DIST_MAX_AU = 0.5: la CAD API fija dist-max en 0.05 au POR DEFECTO, que es
        # exactamente el umbral de distancia de la definicion PHA. Dejarlo implicito
        # censura la muestra en el umbral de la propia etiqueta: satura MOID<=0.05
        # (99.7% de los objetos) y empobrece H<=22, de modo que PHA colapsa a un
        # unico umbral. 0.5 au es el limite superior que sirve JPL.
        df = datos.descargar_cad(dist_max=DIST_MAX_AU)
        print(f"✔ Descarga completada. {len(df):,} registros recibidos.")
        print(f"  Columnas disponibles: {list(df.columns)}")

        # Seleccion de columnas, conversion numerica, imputacion del diametro
        # desde H y renombrado al esquema del CSV del proyecto.
        df = datos.preparar_cad(df)

        df.to_csv(ruta_csv, index=False)
        print(f"✔ CSV guardado en {ruta_csv}")

        df['Close-Approach (CA) Date'] = df['Close-Approach (CA) Date'].apply(datos.limpiar_fecha)

    except requests.exceptions.RequestException as e:
        print(f"✘ Error de conexión: {e}")
        raise
    except Exception as e:
        print(f"✘ Error al procesar los datos: {e}")
        raise

if df is None:
    raise Exception("No se pudieron cargar los datos. Verifica la conexión o el archivo.")

print("\n           RESUMEN DE DATOS CARGADOS           ")
print(f"  Total de registros          : {len(df):,}")
print(f"  Nulos en H(mag)             : {df['H(mag)'].isna().sum():,}")
print(f"  Nulos en Diameter(km)       : {df['Diameter(km)'].isna().sum():,}")

print("\n            PRIMEROS 5 REGISTROS     ")
print(df[['Object', 'Diameter(km)', 'CA DistanceNominal (au)']].head().to_string(index=False))

if not necesita_descarga:
    print(f"\n Fuente: archivo local  →  {ruta_csv}")
    print(f"   Última modificación: {time.ctime(os.path.getmtime(ruta_csv))}")
else:
    print(f"\n Fuente: API de JPL (descarga fresca)")

✔ Archivo CSV encontrado en .\close_approaches.csv
✔ Archivo actualizado (menos de 30 días). Cargando desde archivo local...
✔ Datos cargados correctamente. 32,568 registros.

           RESUMEN DE DATOS CARGADOS           
  Total de registros          : 32,568
  Nulos en H(mag)             : 8
  Nulos en Diameter(km)       : 7

            PRIMEROS 5 REGISTROS     
    Object  Diameter(km)  CA DistanceNominal (au)
    509352      0.333013                 0.009632
2014 SC324      0.047039                 0.039964
2012 UK171      0.045547                 0.049706
  2024 BA5      0.022206                 0.026434
  2024 BW1      0.033609                 0.037979

 Fuente: archivo local  →  .\close_approaches.csv
   Última modificación: Thu Jun 18 20:00:07 2026


## Etiquetado de peligrosidad (PHA)

El objetivo del proyecto es **inferir el carácter potencialmente peligroso (PHA) de un NEO a partir únicamente de la cinemática de sus aproximaciones observadas**, sin usar los elementos orbitales que definen formalmente la etiqueta. Por eso construimos el *target* en dos versiones:

- **`PHA_official`** — el flag oficial `pha` de la [Small-Body Database (SBDB)](https://ssd-api.jpl.nasa.gov/doc/sbdb_query.html) de JPL. Es el *ground truth*. Un objeto es PHA si su **MOID ≤ 0.05 au** *y* su **magnitud absoluta H ≤ 22** (diámetro ≳ 140 m)[^1]. Traemos además `MOID` y `H` oficiales **solo para validar y etiquetar**, nunca como variables predictoras.
- **`PHA_proxy`** — etiqueta derivada *exclusivamente de los datos observados*: el objeto se marca peligroso si su **H observado ≤ 22** y la **mínima distancia de aproximación observada (`dist_min`) ≤ 0.05 au**, usando la distancia observada como aproximación del MOID. La comparación `proxy` vs `official` mide cuán bien la distancia observada sustituye al MOID orbital (sub-resultado del paper).

> **Caveat documentado:** el diámetro fue imputado desde H con la relación de albedo (`albedo = 1329/√0.14`)[^2] en la celda anterior cuando faltaba; por eso `H(mag)` y `Diameter(km)` no son independientes para esos registros. Las etiquetas se calculan a nivel de **objeto** (no de evento) y se almacenan denormalizadas en el mismo CSV para no alterar la estructura de un solo archivo del pipeline.

> **Metadatos de selección (no son features):** también traemos de la SBDB la **fecha real de primera observación** (`first_obs`), el **arco orbital** (`data_arc`) y el número de observaciones (`n_obs_used`). Permiten un análisis de la función de selección observacional riguroso (por fecha de descubrimiento, no por año del primer evento de aproximación) y medir el confundidor de caracterización orbital.

[^1]: Criterio oficial de Potentially Hazardous Asteroid. Fuente: [CNEOS FAQ](https://cneos.jpl.nasa.gov/faq/) (JPL/NASA, Center for Near Earth Object Studies).
[^2]: Relación estándar H–albedo–diámetro `D = 1329·10^(-0.2H)/√p_V`. Fuente: [CNEOS Asteroid Size Estimator](https://cneos.jpl.nasa.gov/tools/ast_size_est.html) (JPL/NASA), que cita Bowell et al. (1989), *Asteroids II*, pp. 524-556, y Harris & Harris (1997), *Icarus* 126:450-454.

In [5]:
# Catalogo SBDB: cache local reciente o descarga (ver neos.datos.descargar_sbdb)
sbdb = datos.normalizar_sbdb(datos.descargar_sbdb(RUTA_SBDB))

# --- Union por designacion (Object == pdes) ---
df["Object"] = df["Object"].astype(str)
mapa = sbdb.drop_duplicates("pdes").set_index("pdes")

# Columnas oficiales: etiqueta + validacion + metadatos de SELECCION.
# NINGUNA se usa como feature predictora (eso seria circular).
df["MOID (au)"]      = df["Object"].map(mapa["moid"])
df["H_SBDB(mag)"]    = df["Object"].map(mapa["H"])
df["first_obs_year"] = df["Object"].map(mapa["first_obs_year"])  # fecha REAL de 1a observacion
df["data_arc(d)"]    = df["Object"].map(mapa["data_arc"])        # arco orbital (caracterizacion)
df["n_obs_used"]     = df["Object"].map(mapa["n_obs_used"])      # nro de observaciones astrometricas
df["PHA_official"]   = df["Object"].map(mapa["pha01"]).astype("Int8")

# --- Evento OBSERVADO vs calculado retroactivamente ---
# La CAD API integra hacia atras hasta 1900 tambien para objetos descubiertos
# despues: esos eventos son salidas de un modelo dinamico, no observaciones.
# Se marcan para poder restringir el analisis principal a lo realmente
# observado y dejar el catalogo completo para el anexo de sensibilidad.
_ca_year = pd.to_datetime(df["Close-Approach (CA) Date"].apply(datos.limpiar_fecha),
                          format="%Y-%b-%d %H:%M", errors="coerce").dt.year
df["post_discovery"] = ((_ca_year >= df["first_obs_year"])
                        .mask(df["first_obs_year"].isna()).astype("Int8"))

# --- Etiqueta PROXY: propiedades por OBJETO desde lo realmente OBSERVADO ---
# Solo eventos post-descubrimiento: incluir los integrados inflaria la calidad
# aparente del proxy (corr con el MOID sube de 0.868 a 0.931 si se incluyen).
_obs = df[df["post_discovery"] == 1]
distmin_obj = df["Object"].map(_obs.groupby("Object")["CA DistanceMinimum (au)"].min())
H_obj       = df["Object"].map(_obs.groupby("Object")["H(mag)"].min())
df["PHA_proxy"] = datos.etiqueta_proxy(H_obj, distmin_obj).astype("Int8")

# --- Guardar CSV enriquecido (un unico archivo, estructura intacta) ---
df.to_csv(ruta_csv, index=False)
print(f"✔ CSV actualizado con etiquetas en {ruta_csv}")

# --- Snapshot CONGELADO y fechado para reproducibilidad del paper ---
# Se congela UNA sola vez: si ya existe un close_approaches_v*.csv no se crea
# otro ni se toca snapshot_info.json (protege el snapshot citado en el paper).
# Para re-congelar con datos nuevos: borrar los close_approaches_v*.csv.
import json as _json, sys as _sys, glob as _glob
_snaps = sorted(_glob.glob(os.path.join(DIR_DATOS, "close_approaches_v*.csv")))
if _snaps:
    print(f"✔ Snapshot ya congelado: {_snaps[-1]}  (no se sobrescribe; ver snapshot_info.json)")
else:
    fecha_snap = datetime.utcnow().strftime("%Y%m%d")
    ruta_snap = os.path.join(DIR_DATOS, f"close_approaches_v{fecha_snap}.csv")
    df.to_csv(ruta_snap, index=False)
    _info = {"snapshot_date_utc": fecha_snap,
             # dist_max_au queda registrado: es el parametro cuyo valor por
             # defecto (0.05) censuraba la muestra en el umbral de la etiqueta.
             "dist_max_au": DIST_MAX_AU,
             "n_events": int(len(df)),
             "n_events_post_discovery": int((df["post_discovery"] == 1).sum()),
             "n_objects": int(df["Object"].nunique()),
             "n_pha_official": int((df.drop_duplicates("Object")["PHA_official"] == 1).sum()),
             "python": _sys.version.split()[0], "pandas": pd.__version__}
    with open(os.path.join(DIR_DATOS, "snapshot_info.json"), "w") as _fj:
        _json.dump(_info, _fj, indent=2)
    print(f"✔ Snapshot congelado: {ruta_snap}  (ver snapshot_info.json)")

# --- Resumen / validacion fisica (a nivel OBJETO) ---
obj = df.drop_duplicates("Object")
n_match = int(obj["PHA_official"].notna().sum())
print("\n        RESUMEN DE ETIQUETADO (por objeto)        ")
print(f"  Objetos unicos              : {len(obj):,}")
print(f"  Emparejados con SBDB        : {n_match:,} ({100*n_match/len(obj):.1f}%)")
print(f"  PHA_official = 1            : {int((obj['PHA_official']==1).sum()):,}")
print(f"  PHA_proxy    = 1            : {int((obj['PHA_proxy']==1).sum()):,}")

n_obs_ev = int((df["post_discovery"] == 1).sum())
print(f"  Eventos observados          : {n_obs_ev:,} de {len(df):,} "
      f"({100*n_obs_ev/len(df):.1f}%; el resto son integrados hacia atras)")

val = obj.dropna(subset=["PHA_official"])
tp = int(((val["PHA_proxy"]==1) & (val["PHA_official"]==1)).sum())
fp = int(((val["PHA_proxy"]==1) & (val["PHA_official"]==0)).sum())
fn = int(((val["PHA_proxy"]==0) & (val["PHA_official"]==1)).sum())
prec = tp/(tp+fp) if tp+fp else float("nan")
rec  = tp/(tp+fn) if tp+fn else float("nan")
print(f"  Proxy vs oficial           : precision={prec:.3f}  recall={rec:.3f}")
# Correlacion proxy-MOID por objeto, solo sobre eventos observados
_dmin_obs = _obs.groupby("Object")["CA DistanceMinimum (au)"].min()
_vm = pd.DataFrame({"dmin": _dmin_obs,
                    "moid": mapa["moid"].reindex(_dmin_obs.index)}).dropna()
print(f"  corr(dist_min observ., MOID): {_vm['dmin'].corr(_vm['moid']):.3f}")

✔ Catalogo SBDB local reciente. Cargando .\sbdb_neo.csv...


✔ CSV actualizado con etiquetas en .\close_approaches.csv
✔ Snapshot ya congelado: .\close_approaches_v20260619.csv  (no se sobrescribe; ver snapshot_info.json)

        RESUMEN DE ETIQUETADO (por objeto)        
  Objetos unicos              : 18,934
  Emparejados con SBDB        : 18,927 (100.0%)
  PHA_official = 1            : 1,379
  PHA_proxy    = 1            : 1,401
  Proxy vs oficial           : precision=0.968  recall=0.983
  corr(dist_min observ., MOID): 0.725


---

## Datos listos

El dataset procesado ha sido guardado como `close_approaches.csv` en esta misma carpeta (`data/`).

Para continuar con el analisis estadistico y machine learning, ejecutar el notebook `ProyectoNeoRework_ml.ipynb`.